# Jonav: SimCSE supervisado en Google Colab

Tu rama local es `jonav`. Este notebook usa el código incluido en
**jonav_colab_fuentes.zip**, sin necesitar que publiques la rama primero.

1. En Colab, sube este `.ipynb` desde **Archivo > Subir notebook**.
2. Selecciona una **GPU** en el tipo de entorno de ejecución.
3. Ejecuta la instalación y sube el ZIP cuando se solicite.
4. Reinicia la sesión de Python después de instalar; sigue desde la sección 2.
5. Ejecuta las secciones en orden. Revisa la selección antes de habilitar test.

El cálculo corre en Colab. Drive conserva checkpoints, configuraciones y métricas.
Colab puede desconectarse: las ejecuciones terminadas se reutilizan; una ejecución
interrumpida comienza de nuevo. No se promete reanudar el optimizador a mitad de época.
No necesitas un token de Hugging Face para entrenar.

Alcance: 6 configuraciones con negativos, selección con dev, ablación con/sin
negativos y comparación de dev entre 3 semillas. Test se usa al cerrar los dos
modelos de la comparación, sin elegir parámetros a partir de él.

## 1. Subir las fuentes e instalar

Sube **jonav_colab_fuentes.zip**. Se conserva PyTorch con CUDA del runtime.
Después de esta celda, **reinicia la sesión de Python** para cargar las versiones
instaladas. No vuelvas a ejecutar esta celda después del reinicio.

In [ ]:
import os, sys, subprocess, zipfile, io
from pathlib import Path
from google.colab import files

uploaded = files.upload()
archive_name = "jonav_colab_fuentes.zip"
if archive_name not in uploaded:
    raise ValueError("Sube jonav_colab_fuentes.zip junto con este notebook.")
target = Path("/content").resolve()
with zipfile.ZipFile(io.BytesIO(uploaded[archive_name])) as archive:
    for member in archive.infolist():
        candidate = (target / member.filename).resolve()
        if not candidate.is_relative_to(target):
            raise ValueError("Ruta inválida en el ZIP.")
    archive.extractall(target)
repo = target / "Sentence-embedding-model"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
print("Python:", sys.version.split()[0], flush=True)
print("Instalando dependencias; el progreso y los errores aparecerán aquí.", flush=True)
command = [sys.executable, "-m", "pip", "install", "--only-binary=:all:",
           "--disable-pip-version-check", "--progress-bar", "off", "-r",
           str(repo / "requirements-colab.txt")]
with (repo / "installation.log").open("w", encoding="utf-8") as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()
    returncode = process.wait()
if returncode:
    raise RuntimeError(f"Falló la instalación (código {returncode}). Revisa las líneas anteriores y installation.log.")
print("Instalación terminada. Reinicia la sesión de Python y continúa en la sección 2.")

## 2. GPU, Drive y pruebas del código

Autoriza el montaje de Drive. Tus archivos quedarán en
`Mi unidad/SimCSE_jonav/`. El código y el dataset se leen desde el disco de Colab;
los checkpoints y JSON se escriben en Drive.

In [ ]:
import os, sys, json, gc, subprocess, shutil, hashlib
from pathlib import Path
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
import torch
from google.colab import drive

REPO = Path("/content/Sentence-embedding-model")
assert REPO.exists(), "Ejecuta primero la sección 1."
os.chdir(REPO)
sys.path.insert(0, str(REPO))
assert torch.cuda.is_available(), "Selecciona un runtime GPU antes de entrenar."
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/SimCSE_jonav")
RUNS = ROOT / "runs"
CHECKPOINTS = ROOT / "checkpoints"
for directory in [ROOT, RUNS, CHECKPOINTS]:
    directory.mkdir(parents=True, exist_ok=True)
freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(ROOT / "requirements-runtime.txt").write_text(freeze)
shutil.copy2(REPO / "source_manifest.json", ROOT / "source_manifest.json")
for command in [[sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
                [sys.executable, "scripts/validate_dataset.py", "--data", "snli_train_100k.jsonl"]]:
    result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(result.stdout, flush=True)
    result.check_returncode()
print("Salidas persistentes:", ROOT)

## 3. Verificar la evaluación antes de entrenar

Se evalúa BERT sin entrenar, con mean pooling, **solo en dev**. La referencia
de la consigna es 59.31 (Spearman ×100). Si difiere más de 3 puntos, se detiene
el flujo para revisar la evaluación; no se fuerza ni se reemplaza la métrica.

In [ ]:
from src.evaluate import run_evaluation

baseline_file = RUNS / "jonav_sanity_raw_bert_dev.json"
if baseline_file.exists():
    baseline = json.loads(baseline_file.read_text())
else:
    baseline = run_evaluation("bert-base-uncased", model_type="raw_bert",
                              pooling="mean", split="dev", output_json=str(baseline_file))
actual = baseline["dev"]["spearman"]
print("Dev BERT:", actual, "Referencia: 59.31")
assert abs(actual - 59.31) <= 3.0, "Revisa la evaluación antes del entrenamiento."
SANITY_PASSED = True
gc.collect()
torch.cuda.empty_cache()

## 4. Configuración y prueba corta

Valores iniciales del equipo: CLS + MLP, batch 64, 3 épocas, semilla 42,
longitud máxima 64. La MLP entrenada se conserva al evaluar el modo supervisado,
como indica el [repositorio oficial de SimCSE](https://github.com/princeton-nlp/SimCSE#evaluation).

La prueba usa un batch para detectar falta de memoria, pérdida no finita y
problemas al recargar el checkpoint. **Si no cabe, cambia BATCH_SIZE y repite
esta sección antes de cualquier ejecución completa.** Usa el mismo batch en
toda la búsqueda y ablación. No se sustituye la acumulación de gradientes por
un batch contrastivo mayor: no produciría los mismos negativos.

In [ ]:
from argparse import Namespace
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from src.data import build_supervised_dataset, SupervisedSimCSEDataset, SupDataCollator
from src.models import SimCSEModel
from src.train import train, set_seed

BATCH_SIZE = 64
EPOCHS = 3
SEEDS = [42, 123, 456]
LEARNING_RATES = [3e-5, 5e-5]
TEMPERATURES = [0.03, 0.05, 0.1]
assert SANITY_PASSED
set_seed(42)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
triplets = build_supervised_dataset("snli_train_100k.jsonl")
assert len(triplets) == 33351
print("Pares:", len(triplets), "Con contradicción:", sum(t["negative"] is not None for t in triplets))
loader = DataLoader(SupervisedSimCSEDataset(triplets[:BATCH_SIZE]),
                    batch_size=BATCH_SIZE, collate_fn=SupDataCollator(tokenizer))
batch = next(iter(loader))
model = SimCSEModel(inference_mlp=True).to("cuda").train()
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
def to_gpu(inputs):
    return {k: v.to("cuda") for k, v in inputs.items()} if inputs is not None else None
loss, logits = model.forward_supervised(to_gpu(batch["premises"]),
                                         to_gpu(batch["positives"]), to_gpu(batch["negatives"]))
assert torch.isfinite(loss)
loss.backward()
optimizer.step()
torch.cuda.synchronize()
print("Pérdida de prueba:", loss.item(), "Logits:", tuple(logits.shape))
print("Pico de VRAM (GB):", round(torch.cuda.max_memory_allocated() / 2**30, 2))
inputs = tokenizer(["A dog runs in the park."], return_tensors="pt").to("cuda")
before = model.get_sentence_embeddings(**inputs).cpu()
smoke_dir = CHECKPOINTS / "smoke_only"
model.save_checkpoint(str(smoke_dir), tokenizer)
del loss, logits, optimizer, model, batch, inputs
gc.collect()
torch.cuda.empty_cache()
restored = SimCSEModel.from_checkpoint(str(smoke_dir), device="cuda")
inputs = tokenizer(["A dog runs in the park."], return_tensors="pt").to("cuda")
torch.testing.assert_close(before, restored.get_sentence_embeddings(**inputs).cpu())
del restored, inputs, before, loader
gc.collect()
torch.cuda.empty_cache()
SMOKE_PASSED = True
print("Prueba corta y recarga correctas. Ya puedes entrenar.")

## 5. Entrenar seis configuraciones y seleccionar con dev

Esta es la primera sección de entrenamiento completo. Cada configuración
guarda su mejor checkpoint según dev cada 250 pasos y al terminar una época.
Puede tardar varias horas según la GPU asignada. No se calcula test aquí.
Al volver a ejecutar, se omiten las configuraciones ya terminadas.

In [ ]:
assert SANITY_PASSED and SMOKE_PASSED

def fit(lr, temperature, seed=42, hard_negatives=True):
    name = (f"jonav_sup_hn{'on' if hard_negatives else 'off'}_lr{lr:g}"
            f"_tau{temperature:g}_bs{BATCH_SIZE}_ep{EPOCHS}_seed{seed}")
    result_path = RUNS / f"{name}.json"
    if result_path.exists():
        result = json.loads(result_path.read_text())
        if Path(result["checkpoint_path"], "simcse_config.json").exists():
            print("Reutilizando:", name)
            return result
    args = Namespace(mode="sup", data_path="snli_train_100k.jsonl",
        model_name_or_path="bert-base-uncased", output_dir=str(CHECKPOINTS),
        runs_dir=str(RUNS), run_name=name, pooling="cls", lr=lr,
        batch_size=BATCH_SIZE, epochs=EPOCHS, temperature=temperature,
        seed=seed, dropout_rate=None, weight_decay=0.0, warmup_ratio=0.05,
        max_length=64, eval_steps=250, eval_batch_size=32,
        same_dropout_mask=False, no_hard_negatives=not hard_negatives, eval_test=False)
    try:
        return train(args)
    finally:
        gc.collect()
        torch.cuda.empty_cache()

trials = []
for lr in LEARNING_RATES:
    for temperature in TEMPERATURES:
        trials.append(fit(lr, temperature))
best = max(trials, key=lambda r: r["results"]["dev"]["spearman"])
selection = {"selected_using": "dev_only", "best_run": best["run_name"],
             "lr": best["hyperparameters"]["lr"],
             "temperature": best["hyperparameters"]["temperature"],
             "batch_size": BATCH_SIZE, "epochs": EPOCHS,
             "trial_runs": [r["run_name"] for r in trials]}
(ROOT / "selection_dev.json").write_text(json.dumps(selection, indent=2))
for r in trials:
    print(r["run_name"], "Dev:", r["results"]["dev"]["spearman"])
print("Seleccionado usando solo dev:", best["run_name"])

## 6. Ablación y variación entre semillas

Se usa la configuración ganadora en dos modos: negativos ON y OFF.
En cada pareja cambia únicamente la inclusión de contradicciones; ambos
modos usan todos los mismos pares de entailment. Las semillas adicionales
estiman la variación de dev y no participan en la selección del checkpoint
final de semilla 42. Una muestra de 3 semillas aporta evidencia descriptiva,
sin demostrar significación estadística.

In [ ]:
import numpy as np
lr, temperature = selection["lr"], selection["temperature"]
paired = []
for seed in SEEDS:
    on = fit(lr, temperature, seed, True)
    off = fit(lr, temperature, seed, False)
    paired.append({"seed": seed, "on": on, "off": off,
                   "delta_dev_on_minus_off": on["results"]["dev"]["spearman"] - off["results"]["dev"]["spearman"]})
noise = {"seeds": SEEDS,
         "on_dev_std": float(np.std([p["on"]["results"]["dev"]["spearman"] for p in paired], ddof=1)),
         "off_dev_std": float(np.std([p["off"]["results"]["dev"]["spearman"] for p in paired], ddof=1)),
         "paired_delta_dev_mean": float(np.mean([p["delta_dev_on_minus_off"] for p in paired])),
         "paired_delta_dev_std": float(np.std([p["delta_dev_on_minus_off"] for p in paired], ddof=1)),
         "pairs": [{"seed": p["seed"], "on": p["on"]["run_name"], "off": p["off"]["run_name"],
                    "delta_dev_on_minus_off": p["delta_dev_on_minus_off"]} for p in paired]}
(ROOT / "ablation_dev_noise.json").write_text(json.dumps(noise, indent=2))
print(json.dumps(noise, indent=2))

## 7. Cerrar la selección y evaluar test una vez

Revisa las secciones anteriores. Cuando hayas terminado los ajustes, cambia
`FINALIZE` a `True` y ejecuta esta celda. Evalúa una vez los dos modelos
de semilla 42 y conserva sus resultados. Si vuelves a ejecutarla, lee los JSON
existentes. Una vez calculado test, no cambies la selección para mejorar ese número.

In [ ]:
FINALIZE = False
assert FINALIZE, "Termina los ajustes y establece FINALIZE=True para la evaluación final."
final_pair = next(p for p in paired if p["seed"] == 42)
locked_file = ROOT / "selection_locked.json"
locked = {"on": final_pair["on"]["run_name"], "off": final_pair["off"]["run_name"]}
if locked_file.exists():
    assert json.loads(locked_file.read_text()) == locked, "La selección final ya está cerrada."
else:
    locked_file.write_text(json.dumps(locked, indent=2))

final_results = {}
for label in ["on", "off"]:
    run = final_pair[label]
    path = RUNS / f"{run['run_name']}_final_test.json"
    if path.exists():
        final_results[label] = json.loads(path.read_text())
    else:
        final_results[label] = run_evaluation(run["checkpoint_path"], model_type="simcse",
                                             pooling="cls", split="test", output_json=str(path))
    run["results"]["test"] = final_results[label]["test"]
    (RUNS / f"{run['run_name']}.json").write_text(json.dumps(run, indent=2))
history_path = RUNS / "run_history.json"
history = json.loads(history_path.read_text())
for record in history:
    for label in ["on", "off"]:
        if record["run_name"] == final_pair[label]["run_name"]:
            record["results"]["test"] = final_results[label]["test"]
history_path.write_text(json.dumps(history, indent=2))
print("Test conservado en Drive. No se usa para elegir parámetros.")

## 8. Tabla y evidencia para tu reporte

Se generan CSV y Markdown con configuración, métricas, deltas y variación
de dev. Añade tu interpretación: qué cambió con los negativos y por qué;
compara el resultado supervisado con el artículo junto con tu equipo.
Los checkpoints están en Drive y no deben subirse a Git como archivos binarios.

In [ ]:
import pandas as pd
rows = []
for label in ["on", "off"]:
    run = final_pair[label]
    for split in ["dev", "test"]:
        rows.append({"hard_negatives": label, "split": split,
                     "seed": 42, **run["results"][split]})
table = pd.DataFrame(rows)
table.to_csv(ROOT / "supervised_benchmark.csv", index=False)
delta_dev = final_pair["delta_dev_on_minus_off"]
delta_test = final_results["on"]["test"]["spearman"] - final_results["off"]["test"]["spearman"]
lines = ["# Resultados supervisados de Jonav", "",
         f"Configuración: {json.dumps(selection, ensure_ascii=False)}", "",
         "| Negativos | Split | Spearman x100 | Alignment | Uniformity |",
         "|---|---|---:|---:|---:|"]
for row in rows:
    lines.append(f"| {row['hard_negatives']} | {row['split']} | {row['spearman']:.2f} | {row['alignment']:.4f} | {row['uniformity']:.4f} |")
lines.extend(["", f"Delta ON - OFF: dev {delta_dev:+.2f}; test {delta_test:+.2f} puntos.",
              f"Media de deltas dev entre semillas: {noise['paired_delta_dev_mean']:+.2f}; desviación: {noise['paired_delta_dev_std']:.2f}.",
              "", "Alignment: pares STS-B con puntuación humana >=4/5. Uniformity: todas las oraciones de cada split.",
              "", "Pendiente de redactar: interpretación, limitaciones y distancia al artículo."])
(ROOT / "JONAV_RESULTADOS.md").write_text("\n".join(lines), encoding="utf-8")
print(table.to_string(index=False))
print("Entregables:", ROOT)

## 9. Descargar evidencia para integrar al repo

El ZIP contiene JSON, CSV, informe y versiones exactas del entorno.
Se verifica y guarda en `Mi unidad/SimCSE_jonav/jonav_resultados.zip`.
Descárgalo desde Drive: clic derecho en el ZIP y **Descargar**.
Los checkpoints permanecen en Drive para compartirlos con Rivaldo y Russel,
quienes llevan evaluación y publicación. Este notebook no publica en GitHub
ni Hugging Face automáticamente.

In [ ]:
import zipfile, shutil
from pathlib import Path

root = Path("/content/drive/MyDrive/SimCSE_jonav")
archive = Path("/content/jonav_resultados.zip")
drive_archive = root / archive.name
required = [root / "supervised_benchmark.csv", root / "JONAV_RESULTADOS.md", root / "ablation_dev_noise.json"]
assert all(p.is_file() for p in required), "Faltan archivos de evidencia de las secciones anteriores."
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(root.rglob("*")):
        if path.is_file() and "checkpoints" not in path.relative_to(root).parts and path != drive_archive:
            bundle.write(path, path.relative_to(root).as_posix())
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None, "El ZIP no pasó la verificación."
    file_count = len(bundle.namelist())
shutil.copy2(archive, drive_archive)
assert drive_archive.stat().st_size == archive.stat().st_size
print("ZIP verificado y guardado en Drive:", drive_archive)
print("Archivos:", file_count, "| Bytes:", drive_archive.stat().st_size)
print("Descárgalo desde Mi unidad > SimCSE_jonav > jonav_resultados.zip")